In [1]:
import torch
from ultralytics import YOLO

# Bu ablasyon SADECE P2 dalini olcer - SPD-Conv YOK.
# Dolayisiyla spdconv import'u / register_spdconv() gerekmiyor;
# p2.yaml tamamen standart moduller kullaniyor (Conv/C3k2/Concat/Upsample).

In [2]:
device = 0 if torch.cuda.is_available() else "cpu" 
print(f"Kullanilan cihaz: {'GPU (cuda:0)' if device == 0 else 'CPU'}")
if device == "cpu":
        print("UYARI: GPU bulunamadi, egitim CPU'da yapilacak ve cok yavas olabilir.")

Kullanilan cihaz: GPU (cuda:0)


In [3]:
# --- P2 ablasyonu (SADECE P2) ---
# Baseline'a gore TEK degisiklik: stride-4 P2 tespit dali eklendi.
# Detect 3 -> 4 seviye [P2/4, P3/8, P4/16, P5/32].
# Backbone ve tum downsample'lar duz Conv, yani baseline ile birebir ayni.
#
# P2 NEDEN: modules_v2.py'deki etiket istatistigine gore valid/test'in
# ~%38-39'u kucuk nesne (<32px). Stride-4 dali tam bu dagilimi hedefliyor.
#
# AGIRLIK TRANSFERI: 360/902 tensor, %60.3 parametre.
# Backbone (0-10) ve head'in ilk kismi (11-16) baseline ile AYNI indekste
# ve ayni sekilde oldugu icin birebir transfer oluyor. 17. katmandan
# itibaren P2 kolu basliyor; orasi ve 4 seviyeli Detect basi rastgele
# baslar - beklenen durum.
#
# KIYAS NOKTALARI:
#   baseline (P2 yok)          -> 2.50M parametre
#   bu kosu (P2)               -> 2.52M parametre
#   SPD-Conv + P2              -> 4.55M parametre, test mAP50-95 %64.93
# P2 tek basina neredeyse bedava (+0.02M); asil maliyet SPD-Conv'dan geliyordu.
model = YOLO("p2.yaml").load("../baseline/runs/detect/train/weights/best.pt")

WARNING no model scale passed. Assuming scale='n'.
Transferred 360/902 items from pretrained weights


In [ ]:
# --- P2 egitimi ---
# MALIYET: parametre neredeyse degismiyor (2.50M -> 2.52M) ama P2 dali
# stride-4 seviyesinde calisiyor, yani P3'un 4 kati AKTIVASYON alani.
# VRAM ve epoch suresi buradan artar (parametreden degil).
# CUDA OOM alirsan batch=8 yap.
model.train(
    data="../../../dataset2/yolo26/YOLO.v1i.yolo26/data.yaml",
    epochs=500,
    imgsz=640,
    batch=16,
    optimizer="SGD",
    device=device,
    patience=100,
    plots=True,
    name="ablation_p2_v",
)

New https://pypi.org/project/ultralytics/8.4.137 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.12.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../../dataset2/yolo26/YOLO.v1i.yolo26/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=p2.yaml, momentum=0.937, mosaic=1.0, multi_sc

In [ ]:
import csv, glob
run = sorted(glob.glob("runs/detect/ablation_p2_v/results.csv"))[-1]
rows = list(csv.DictReader(open(run)))
print(run, "| epoch:", len(rows))
for r in rows[:2] + rows[-3:]:
    print("ep %-4s box=%-8s cls=%-8s mAP50=%-7s mAP50-95=%s" % (
        r["epoch"], r["train/box_loss"], r["train/cls_loss"],
        r["metrics/mAP50(B)"], r["metrics/mAP50-95(B)"]))

b = max(rows, key=lambda r: float(r["metrics/mAP50-95(B)"]))
print("\nEN IYI epoch %s -> P %.3f R %.3f mAP50 %.4f mAP50-95 %.4f" % (
    b["epoch"], float(b["metrics/precision(B)"]), float(b["metrics/recall(B)"]),
    float(b["metrics/mAP50(B)"]), float(b["metrics/mAP50-95(B)"])))
print("KIYAS  SPD-Conv+P2 (valid en iyi)   mAP50 0.8885 mAP50-95 0.6427")

box1 = float(rows[0]["train/box_loss"])
print("\n[saglik] 1. epoch train/box_loss = %.2f -> %s" % (
    box1, "OK" if box1 < 20 else "!! KAYIP OLCEGI PATLAMIS, DURDUR"))
if len(rows) < 500:
    print("[uyari] kosu %d epoch'ta bitmis, 500 degil - erken durdurma (patience) ya da yarida kesilme" % len(rows))

In [ ]:
from pathlib import Path
best = Path(sorted(glob.glob("runs/detect/ablation_p2_v/weights/best.pt"))[-1])
print("test ediliyor:", best)

res = YOLO(str(best)).val(
    data="../../../dataset2/yolo26/YOLO.v1i.yolo26/data.yaml",
    split="test",
    imgsz=640,
    device=device,
)
d = res.results_dict
print("\n=== TEST ===")
for k, lbl in [("metrics/precision(B)", "Precision"),
               ("metrics/recall(B)", "Recall"),
               ("metrics/mAP50(B)", "mAP@0.5"),
               ("metrics/mAP50-95(B)", "mAP@0.5:0.95")]:
    v = d.get(k)
    print(f"{lbl:>14}: {v*100:.2f}%" if v is not None else f"{lbl:>14}: yok")

try:
    for i, c in enumerate(res.names.values()):
        print(f"  {c:>6}: mAP50={res.box.ap50[i]:.4f}  mAP50-95={res.box.ap[i]:.4f}")
except Exception as e:
    print("sinif bazinda metrik alinamadi:", e)

print("\nKIYAS (ayni test seti, 279 goruntu / 1023 kutu):")
print("  SPD-Conv + P2 : mAP50 88.65%  mAP50-95 64.93%")